# VLM Applications

**Module:** 15 — VLMs & Multimodal

From accessibility to industrial inspection — application patterns and safety controls.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Catalog major VLM application patterns
- Sketch a vision gateway reference architecture
- Apply safety controls for NSFW, PII, biometrics, high stakes
- Rate-limit and audit a public image API


## Application Catalog

### Definition
VLM apps turn visual inputs into decisions, text, structure, or actions in a workflow.

### Why it matters
Differentiation is schema, UX, safety, and integration — not only the base model.

### How it works
Map input source, SLA, HITL policy, ontology; share a vision gateway.

### Intuition
Pattern libraries beat one-off prompts per team.

### Pitfalls
- Medical advice without clinicians
- Storing user images forever without consent

### When to use
Product planning and platform architecture.


### Catalog

| App | Input | Output | Safety |
|-----|-------|--------|--------|
| Alt-text | UGC images | Short text | No identity guessing |
| Doc extract | Scans | JSON | Validate money/IDs |
| Catalog enrich | Product photos | Attributes | Brand tone |
| Chart QA | Plots | Numbers | Cite series |
| UI QA | Screenshots | Bugs | Secret exfil risk |
| Inspection | Defect photos | Pass/fail | Calibrate+HITL |
| Moderation | UGC | Labels | Appeals |
| MM agents | Screen+tools | Actions | Confirm destructive tools |

```mermaid
flowchart TB
  C[Clients] --> G[Vision gateway]
  G --> S[Safety]
  S --> R[Router]
  R --> V1[Mini VLM]
  R --> V2[Frontier]
  R --> O[OCR]
  V1 --> V[Validators]
  V2 --> V
  O --> V --> H{HITL?}
  H -->|yes| HUM[Queue]
  H -->|no| OUT[Response+audit]
```


In [ ]:
# Demo 1: app profiles
APPS = {
    "alt_text": {"sla_ms":2000,"model":"mini","hitl":False,"store_days":7},
    "invoice_extract": {"sla_ms":15000,"model":"frontier+ocr","hitl":"on_low_conf","store_days":90},
    "ugc_moderation": {"sla_ms":1000,"model":"moderation+vlm","hitl":"appeals","store_days":30},
    "inspect_weld": {"sla_ms":5000,"model":"specialist","hitl":True,"store_days":365},
}
def assert_policy(app, store_days):
    cfg=APPS[app]
    if store_days > cfg["store_days"]: return f"deny_store max={cfg['store_days']}"
    return f"ok model={cfg['model']} hitl={cfg['hitl']}"
print(assert_policy("alt_text",30))
print(assert_policy("invoice_extract",30))


## Safety

### Definition
Vision safety spans content risk, privacy (faces/IDs/docs), and capability risk (wrong high-stakes outputs).

### Why it matters
Images are sticky PII; mistakes are visible and sometimes irreversible.

### How it works
Layer MIME/size → malware → classifiers → policy → output filters → audit → retention. Restrict biometrics.

### Intuition
If you wouldn't paste it in public Slack, don't log the raw URL forever.

### Pitfalls
- Prompt-only safety with live tools
- Face ID features without legal review
- No moderation appeals

### When to use
All user-upload and employee-document vision features.


In [ ]:
# Demo 2: safety gate
FORBIDDEN={"sexual_minor","extreme_violence"}
PII={"face_id","passport","credit_card"}
def safety_gate(labels:set[str], purpose:str)->str:
    if labels & FORBIDDEN: return "block"
    if labels & PII and purpose not in {"kyc_authorized","user_private_vault"}: return "redact_or_deny"
    if "medical_diagnosis_request" in labels: return "allow_with_disclaimer_and_hitl"
    if purpose == "biometric_identify": return "deny"
    return "allow"
print(safety_gate({"passport"},"social_alt_text"))
print(safety_gate({"passport"},"kyc_authorized"))
print(safety_gate(set(),"biometric_identify"))


In [ ]:
# Demo 3: gateway handler mock
import time, json, hashlib
def handle(user_id, app, image_bytes, prompt):
    if len(image_bytes) > 10_000_000: return {"error":"too_large"}
    image_hash = hashlib.sha256(image_bytes).hexdigest()[:16]
    result = {"app":app,"user_id":user_id,"image_hash":image_hash,"output":{"caption":"mock"},"model":"mini-vlm"}
    print(json.dumps({k:result[k] for k in result if k!="output"}))
    return result
handle("u1","alt_text",b"fake","describe")


In [ ]:
# Demo 4: rate limiter
from collections import defaultdict
class ImageRateLimiter:
    def __init__(self, max_per_minute=30):
        self.max=max_per_minute; self.counts=defaultdict(int)
    def allow(self, user_id):
        if self.counts[user_id] >= self.max: return False
        self.counts[user_id]+=1; return True
lim=ImageRateLimiter(3)
print([lim.allow("u") for _ in range(4)])


In [ ]:
# Demo 5: EXIF-like metadata strip
def strip_meta(meta: dict) -> dict:
    deny = {"gps","gps_lat","gps_lon","device_serial","owner"}
    return {k:v for k,v in meta.items() if k.lower() not in deny}
print(strip_meta({"width":800,"gps_lat":1.2,"camera":"Phone"}))


In [ ]:
# Demo 6: moderation appeal state machine
ALLOWED = {
    ("labeled","appeal"): "under_review",
    ("under_review","uphold"): "labeled",
    ("under_review","overturn"): "cleared",
}
def transition(state, event):
    return ALLOWED.get((state,event), state)
s="labeled"
for e in ("appeal","overturn"):
    s=transition(s,e); print(e,s)


### Abuse cases (starter)

1. Flood caption API for scraping
2. Upload CSAM / violent content
3. KYC docs posted to alt-text app
4. Prompt injection via screenshot text
5. Exfil secrets from internal UI shots
6. Face re-identification attempts
7. Model extraction via batched queries
8. Copyright mass captioning


### Checklist — Safety review

- [ ] Purpose binding on PII docs
- [ ] Biometric ID legally reviewed
- [ ] Appeals path for moderation
- [ ] Raw image retention minimized
- [ ] Audit logs without raw bytes


### Try it yourself — Apps & safety

1. Extend gateway with malware MIME allow-list.
2. Write 10 abuse cases for your product.
3. Add SLO alerts on block-rate spikes.

**Stretch:** Design retention TTLs per app profile.


### Try it yourself — Architecture

1. Draw sequence diagram for invoice_extract with HITL.
2. Define OpenAPI sketch for POST /v1/vision.


## Knowledge Check

**Q1.** Why hash images in logs?

<details><summary>Answer</summary>

Enables audit/dedup without retaining raw sensitive pixels.

</details>

**Q2.** Why purpose-bind passport processing?

<details><summary>Answer</summary>

Same bytes may be allowed for KYC but must be denied for social alt-text.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `UGC` | User-generated content |
| `NSFW` | Unsafe content category |
| `retention` | How long raw media is stored |
| `vision gateway` | Shared multimodal entry service |
| `HITL` | Human review loop |


## Key Takeaways

- Apps share gateway, router, validators, audit
- Safety = content + privacy + capability risk
- Log hashes/metadata, not eternal raw dumps
- Match model/HITL/retention to risk profile


## Production Incident Patterns — VLM applications

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "VLM applications",
}))


## Mini Case Study — VLM applications

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("VLM applications", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — VLM applications ops

1. Write a one-page runbook section for on-call when VLM applications critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
